### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [4]:
%pip install polygon-api-client -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
BASE_URL = "https://models.github.ai/inference"
MODEL_GPT_4o_MINI = "gpt-4o-mini"
MODEL_NAME = MODEL_GPT_4o_MINI
API_KEY = "github_pat_"

In [2]:
# from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

# load_dotenv(override=True)

In [5]:
from accounts import Account

In [6]:
account = Account.get("Ed")
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [7]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9972.946, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 9.018, "timestamp": "2026-07-09 08:53:05", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-07-09 08:53:05", 10110.946]], "total_portfolio_value": 10110.946, "total_profit_loss": 110.94599999999991}'

In [8]:
account.report()

'{"name": "ed", "balance": 9972.946, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 9.018, "timestamp": "2026-07-09 08:53:05", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-07-09 08:53:05", 10110.946], ["2026-07-09 08:53:10", 10233.946]], "total_portfolio_value": 10233.946, "total_profit_loss": 233.9459999999999}'

In [9]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 9.018,
  'timestamp': '2026-07-09 08:53:05',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [11]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


McpError: Connection closed

In [12]:
mcp_tools

NameError: name 'mcp_tools' is not defined

In [ ]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = MODEL_NAME

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


### Now let's build our own MCP Client

In [ ]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

In [ ]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
context = await read_accounts_resource("ed")
print(context)

In [ ]:
from accounts import Account
Account.get("ed").report()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>